### Try On Small Corpus To Try The Model On It

In [112]:
documents = [
    "Machine learning is a field of artificial intelligence.",
    "Deep learning uses neural networks to learn from data.",
    "Python is a popular programming language.",
    "Natural language processing helps computers understand human language.",
    "Computer vision allows machines to understand images.",
    "Neural networks are widely used in deep learning.",
    "Artificial intelligence is used in many real world applications.",
    "Natural language processing is used for text classification and translation.",
    "Machine learning algorithms can be used for prediction.",
    "Deep learning models require large amounts of training data.",
    "Omar Tawfik Is A Famous Programmer",
    "omar Tawfik Is A python instructor ",
    "graphic desiging is a filed talk about editng phots and videos",
    "graphic desiging is a filed learned in applied arts college",
    'mostafa hefny is the greatest graphic desginer',
    "ramez tamer is enginner and biggest ass in the country",
    "Electrical enginnering fucks ramez tamer",
    'ramez tamer has a small dick'
]

### Text CLeaning And Preprocessing

In [113]:
import re
def text_clean(text):
  text.lower()
  rext = re.sub(r"[^a-zA-Z\s]", "", text)
  tokens = text.split()

  return tokens


### Trying The Preprocess On A Small Text

In [114]:
text = "Machine Learning Is Amazing!"
tokens = text_clean(text)
print(tokens)

['Machine', 'Learning', 'Is', 'Amazing!']


### Downloading GloVe

In [115]:
# !wget http://nlp.stanford.edu/data/glove.6B.zip

### Unzip GloVe

In [116]:
# !unzip /content/glove.6B.zip

### Loading GloVe To Python And Applying It Into Our Corpus

In [117]:
import numpy as np
embeddings = {}
with open("/content/glove.6B.100d.txt",encoding="utf-8") as f:
  for line in f:
    values = line.split()
    word = values[0]
    vector = np.asarray(values[1:],dtype="float32")
    embeddings[word] = vector


In [118]:
print(len(embeddings))

400000


### Converting Word "machine" Into Vector

In [119]:
print(embeddings["machine"])

[-0.65365    0.49419   -0.26245   -0.20722   -0.11413    0.35701
  1.0454     0.21881    0.52769    0.60606    0.42521   -0.65169
  0.15318   -0.14797    0.12651   -0.017124   0.45325    0.37166
 -0.26847   -0.2627     0.43869   -0.016615   0.12714   -0.54708
  0.089084   0.24336   -0.34415    0.0026505 -0.094268   0.056114
  0.46366    0.68786   -0.20631   -0.088003   0.32153   -0.91399
 -0.080976  -0.90761    0.92889   -0.68033    0.23801   -0.37469
 -0.43278   -0.19243   -0.23711   -0.73041   -0.50592   -0.30237
  0.0017281 -0.60923   -0.21046    0.47403    0.37333    1.2475
  0.6299    -1.5292    -0.32403    0.59681    0.97994    0.59756
  0.67625    0.28223   -0.26748    1.425     -0.34419    0.25212
  0.3024    -0.26582   -0.22583    0.53783   -0.44439   -0.24281
  0.38001    0.085317   0.49694    0.24058    0.20611    0.023896
 -0.53078    0.12086    1.1627    -0.0053908 -0.66132    0.073666
 -1.5987     0.3626     0.68496   -0.93403    0.30523   -0.1688
  0.43895    0.73641    

### Converting ALl Corpus Into embeddings  

In [120]:
def document_vector(tokens, embeddings):

    vectors = []

    for token in tokens:
        if token in embeddings:
            vectors.append(embeddings[token])

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

In [121]:
document_vectors = []

for document in documents:

    tokens = text_clean(document)

    vector = document_vector(tokens, embeddings)

    document_vectors.append(vector)

In [122]:
query = "machine learning algorithms"
query_tokens = text_clean(query)
query_vector = document_vector(
    query_tokens,
    embeddings
)

In [123]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(
    query_vector.reshape(1, -1),
    vector.reshape(1, -1)
)[0][0]

In [124]:
def semantic_search(query, documents, embeddings, k=3):

    query_tokens = text_clean(query)

    query_vector = document_vector(
        query_tokens,
        embeddings
    )

    scores = []

    for i, document in enumerate(documents):

        document_tokens = text_clean(document)

        doc_vector = document_vector(
            document_tokens,
            embeddings
        )

        score = cosine_similarity(
            query_vector.reshape(1, -1),
            doc_vector.reshape(1, -1)
        )[0][0]

        scores.append((i, score))

    scores.sort(
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for index, score in scores[:k]:
        results.append(
            (documents[index], score)
        )

    return results

In [125]:
query = "ramez tamer"

results = semantic_search(
    query,
    documents,
    embeddings,
    k=5
)

for document, score in results:
    print(f"Score: {score:.4f}")
    print(document)
    print()

Score: 0.9095
Electrical enginnering fucks ramez tamer

Score: 0.0908
omar Tawfik Is A python instructor 

Score: 0.0630
ramez tamer has a small dick

Score: 0.0000
Omar Tawfik Is A Famous Programmer

Score: -0.1436
ramez tamer is enginner and biggest ass in the country

